### Algoritmo de aprendizaje supervisado propuesto Random Forest

In [1]:
import os
from dotenv import load_dotenv

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

load_dotenv("/home/jovyan/work/.env")
MONGO_URI = os.getenv("MONGO_URI")

In [2]:
spark = (
    SparkSession.builder
    .appName("RandomForest_AutoTec")
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .config("spark.mongodb.write.connection.uri", MONGO_URI)
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1")
    .getOrCreate()
)

In [3]:
df = (
    spark.read.format("mongodb")
    .option("database", "proyecto_bigdata")
    .option("collection", "Contenedor_Autos_Limpio")
    .load()
)

print("Registros cargados:", df.count())
df.printSchema()

Registros cargados: 1988
root
 |-- _id: string (nullable = true)
 |-- cat_combustible: integer (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- combustible: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- foto_url: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- kilometraje: double (nullable = true)
 |-- marca: string (nullable = true)
 |-- modelo: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- url: string (nullable = true)
 |-- usuario: string (nullable = true)
 |-- year: integer (nullable = true)



In [4]:
df_modelo = df.withColumn(
    "precio_num",
    regexp_replace(col("precio"), "[^0-9]", "").cast("double")
).withColumn(
    "kilometraje_num",
    regexp_replace(col("kilometraje"), "[^0-9]", "").cast("double")
).withColumn(
    "year_num",
    col("year").cast("int")
)

df_modelo = df_modelo.select(
    "precio_num",
    "kilometraje_num",
    "year_num",
    "marca",
    "modelo",
    "combustible",
    "ciudad"
).dropna()

df_modelo = df_modelo.filter(
    (col("precio_num") >= 1000000) &
    (col("kilometraje_num") >= 0) &
    (col("year_num") >= 1990) &
    (col("year_num") <= 2026)
)

print("Registros finales para modelo:", df_modelo.count())
df_modelo.show(5, False)

Registros finales para modelo: 491
+----------+---------------+--------+-------+-------------------------------+-----------+-----------+
|precio_num|kilometraje_num|year_num|marca  |modelo                         |combustible|ciudad     |
+----------+---------------+--------+-------+-------------------------------+-----------+-----------+
|7.49E7    |1103710.0      |2021    |baic   |X35 1.5 Luxury Fl 4x2 Mt 5p    |bencina    |antofagasta|
|8.0E7     |755000.0       |2017    |changan|Cs75                           |bencina    |santiago   |
|7.19E7    |923000.0       |2018    |changan|Cx70 1.6                       |bencina    |araucania  |
|6.95E7    |876260.0       |2023    |chery  |Arrizo 1.5                     |bencina    |santiago   |
|8.79E7    |442510.0       |2022    |chery  |Tiggo 1.5 Vvt Gls 4x2 Cvt At 5p|bencina    |biobio     |
+----------+---------------+--------+-------+-------------------------------+-----------+-----------+
only showing top 5 rows



In [5]:
categoricas = ["marca", "modelo", "combustible", "ciudad"]

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in categoricas
]

encoders = [
    OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_vec")
    for c in categoricas
]

assembler = VectorAssembler(
    inputCols=[
        "kilometraje_num",
        "year_num",
        "marca_vec",
        "modelo_vec",
        "combustible_vec",
        "ciudad_vec"
    ],
    outputCol="features"
)

In [6]:
train, test = df_modelo.randomSplit([0.8, 0.2], seed=42)

print("Datos entrenamiento:", train.count())
print("Datos prueba:", test.count())

Datos entrenamiento: 418
Datos prueba: 73


In [7]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="precio_num",
    predictionCol="prediction",
    numTrees=50,
    maxDepth=8,
    seed=42
)

pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

modelo_rf = pipeline.fit(train)
predicciones = modelo_rf.transform(test)

In [8]:
evaluator_rmse = RegressionEvaluator(
    labelCol="precio_num",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="precio_num",
    predictionCol="prediction",
    metricName="mae"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="precio_num",
    predictionCol="prediction",
    metricName="r2"
)

rmse = evaluator_rmse.evaluate(predicciones)
mae = evaluator_mae.evaluate(predicciones)
r2 = evaluator_r2.evaluate(predicciones)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2: {r2:.4f}")

RMSE: 16416331.13
MAE: 11680592.65
R2: 0.3635


In [9]:
predicciones.select(
    "precio_num",
    "prediction",
    "kilometraje_num",
    "year_num",
    "marca",
    "modelo",
    "combustible",
    "ciudad"
).show(10, False)

+-----------+--------------------+---------------+--------+---------+---------+-----------+------------+
|precio_num |prediction          |kilometraje_num|year_num|marca    |modelo   |combustible|ciudad      |
+-----------+--------------------+---------------+--------+---------+---------+-----------+------------+
|1.1999997E7|6.945616479607664E7 |440000.0       |2022    |peugeot  |208      |diesel     |puerto montt|
|1.5499997E7|5.337843993862268E7 |990000.0       |2022    |great    |Wall Poer|diesel     |puerto montt|
|1.7999997E7|7.697502536260727E7 |420000.0       |2025    |toyota   |Yaris    |bencina    |puerto montt|
|3.2E7      |5.4155186577816255E7|1800000.0      |2009    |ssangyong|Actyon   |diesel     |quilpue     |
|3.8E7      |7.863319360014172E7 |1100000.0      |2013    |chevrolet|Sail     |bencina    |antofagasta |
|3.93E7     |5.159489693548868E7 |1750660.0      |2011    |suzuki   |Swift    |bencina    |maipu       |
|4.5E7      |7.690126043084483E7 |1390000.0      |2014 

Se propone  Random Forest como modelo supervisado principal para la estimación del precio de vehículos usados, ya que permite trabajar con variables numéricas y categóricas, además de capturar relaciones no lineales entre los datos
.